In [ ]:
# # XGBoost for classification, by hand
# Five students (CGPA, placed or not). We grow XGBoost's first classification tree with NumPy, check every number
# from the Note, and compare with the XGBoost library and with scikit-learn's GradientBoostingClassifier.

In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.ensemble import GradientBoostingClassifier

## 1. The data and the base model (log-odds)

In [ ]:
df = pd.DataFrame({"cgpa": [5.70, 6.25, 7.10, 8.15, 9.60], "placed": [0, 1, 0, 1, 1]})
x, y = df["cgpa"].to_numpy(), df["placed"].to_numpy()

# base model: log of the odds of class 1 (3 placed, 2 not)
z0 = np.log(y.sum() / (len(y) - y.sum()))
# sigmoid turns log-odds into a probability
sigmoid = lambda z: 1 / (1 + np.exp(-z))
df["p1"] = sigmoid(z0)
# residual: actual minus predicted probability
df["res1"] = y - df["p1"]
print("log-odds:", z0, " probability:", sigmoid(z0))
df

## 2. Similarity, gain and output with g = p - y and h = p(1 - p)

In [ ]:
def similarity(r, p, lam=0.0):
    # (sum of residuals)^2 / (sum of p(1-p) + lambda)
    return r.sum() ** 2 / ((p * (1 - p)).sum() + lam)

def output(r, p, lam=0.0):
    # sum of residuals / (sum of p(1-p) + lambda), in log-odds
    return r.sum() / ((p * (1 - p)).sum() + lam)

r, p = df["res1"].to_numpy(), df["p1"].to_numpy()
print("root similarity:", round(similarity(r, p), 6))
xs = np.sort(x)
for t in (xs[1:] + xs[:-1]) / 2:
    m = x < t
    sl, sr = similarity(r[m], p[m]), similarity(r[~m], p[~m])
    print(f"CGPA < {t:.3f}: S_L={sl:.4f} S_R={sr:.4f} gain={sl + sr - similarity(r, p):.4f}"
          f"  (Hessian sums {(p[m]*(1-p[m])).sum():.2f} / {(p[~m]*(1-p[~m])).sum():.2f})")

## 3. The depth-1 tree and its outputs

In [ ]:
t = 7.625
m = x < t
w_left, w_right = output(r[m], p[m]), output(r[~m], p[~m])
print("outputs:", w_left, w_right)

## 4. Stage 2: log-odds + eta x output, then back to probability

In [ ]:
eta = 0.3
df["z2"] = z0 + eta * np.where(m, w_left, w_right)
df["p2"] = sigmoid(df["z2"])
df["res2"] = y - df["p2"]
df.round(4)

## 5. Lambda = 1 for comparison

In [ ]:
print("outputs with lambda=1:", output(r[m], p[m], 1), output(r[~m], p[~m], 1))
print("gain with lambda=1:", similarity(r[m], p[m], 1) + similarity(r[~m], p[~m], 1) - similarity(r, p, 1))

## 6. min_child_weight: the Hessian sum in every leaf must reach it (XGBoost's default is 1)

In [ ]:
h = p * (1 - p)
print("each row's Hessian:", h[0], " total:", h.sum())
smallest_side = [min(h[x < t].sum(), h[x >= t].sum()) for t in (xs[1:] + xs[:-1]) / 2]
print("smallest child Hessian sum for each split:", np.round(smallest_side, 2))
print("any split with both children >= 1?", any(s >= 1 for s in smallest_side))

## 7. Check with the XGBoost library
base_score is a probability for binary:logistic. Leaf values in the dump are already multiplied by eta.

In [ ]:
X = x.reshape(-1, 1)
for mcw in (0, 1):
    clf = xgb.XGBClassifier(n_estimators=1, max_depth=1, learning_rate=0.3, reg_lambda=0, gamma=0,
                            min_child_weight=mcw, base_score=0.6, objective="binary:logistic",
                            tree_method="exact", device="cpu").fit(X, y)
    print(f"min_child_weight={mcw}: log-odds {clf.predict(X, output_margin=True)}")
    print(clf.get_booster().get_dump(with_stats=True)[0])
    if mcw == 0:
        assert np.allclose(clf.predict(X, output_margin=True), df["z2"], atol=1e-5)

## 8. Check with scikit-learn
GradientBoostingClassifier starts from the same log-odds and uses the same leaf formula (lambda = 0).
Its tree chooses the split by squared error of the residuals, not by the Hessian-weighted gain;
here every row has the same p(1-p), so both choose CGPA < 7.625.

In [ ]:
gbc = GradientBoostingClassifier(loss="log_loss", n_estimators=1, learning_rate=0.3, max_depth=1).fit(x.reshape(-1, 1), y)
tree = gbc.estimators_[0, 0].tree_
print("threshold:", tree.threshold[0], " leaf values:", tree.value.ravel()[1:])
print("sklearn log-odds:", gbc.decision_function(x.reshape(-1, 1)))
print("ours:            ", df["z2"].to_numpy())
assert np.allclose(gbc.decision_function(x.reshape(-1, 1)), df["z2"])

## 9. The second tree

In [ ]:
r2, p2 = df["res2"].to_numpy(), df["p2"].to_numpy()
for t in (xs[1:] + xs[:-1]) / 2:
    mm = x < t
    print(f"CGPA < {t:.3f}: gain={similarity(r2[mm], p2[mm]) + similarity(r2[~mm], p2[~mm]) - similarity(r2, p2):.4f}")

In [ ]:
# the library's second tree, and scikit-learn's second tree, for comparison
clf2 = xgb.XGBClassifier(n_estimators=2, max_depth=1, learning_rate=0.3, reg_lambda=0, gamma=0, min_child_weight=0,
                         base_score=0.6, objective="binary:logistic", tree_method="exact", device="cpu").fit(X, y)
print(clf2.get_booster().get_dump(with_stats=True)[1])
gbc2 = GradientBoostingClassifier(loss="log_loss", n_estimators=2, learning_rate=0.3, max_depth=1).fit(X, y)
print("scikit-learn second tree threshold:", gbc2.estimators_[1, 0].tree_.threshold[0])